# Customer churn



first we are gonna read the csv and play with the data

In [1]:
import pandas as pd
import numpy as np


In [2]:
first_data=pd.read_csv("datasets/Customer Churn.csv")
sec_data=pd.read_csv("datasets/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [4]:
print((first_data.shape))
print((sec_data.shape))


(3150, 14)
(7043, 21)


In [5]:
first_data.head()

,Call Failure,Complains,Subscription Length,Charge Amount,Seconds of Use,Frequency of use,Frequency of SMS,Distinct Called Numbers,Age Group,Tariff Plan,Status,Age,Customer Value,Churn
0,8,0,38,0,4370,71,5,17,3,1,1,30,197.640,0
1,0,0,39,0,318,5,7,4,2,1,2,25,46.035,0
2,10,0,37,0,2453,60,359,24,3,1,1,30,1536.520,0
3,10,0,38,0,4198,66,1,35,1,1,1,15,240.020,0
4,3,0,38,0,2393,58,2,33,1,1,1,15,145.805,0


In [7]:
sec_data.head()
sec_data.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [15]:
print(sec_data.Churn.value_counts())

Churn
No     5174
Yes    1869
Name: count, dtype: int64


In [33]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import  StandardScaler


x1,x2,y1,y2=train_test_split(first_data.drop(columns='Churn'),first_data['Churn'],test_size=0.2)

# Candidate models

In [125]:
models = {
    "logistic_regression": LogisticRegression(max_iter=1500, C=1.0),
    "random_forest": RandomForestClassifier(n_estimators=350, min_samples_leaf=2, max_features="sqrt", n_jobs=-1),
    "hist_gradient_boosting": HistGradientBoostingClassifier(max_iter=250, learning_rate=0.06, max_leaf_nodes=15, l2_regularization=1.0)
}
def build_pipeline(model):
    return Pipeline([("scaler",StandardScaler()), ("model", model)])

# Compare models

Accuracy is the competition metric . The highest validation score determines the model used for the final fit.

In [126]:
from sklearn.metrics import accuracy_score


validation_results=[]
for name, model in models.items():
    candidate=build_pipeline(model)
    candidate.fit(x1,y1)
    score=accuracy_score(y2,candidate.predict(x2))
    validation_results.append({"model":name,"validation_results":score})
results=pd.DataFrame(validation_results).sort_values("validation_results",ascending=False).reset_index(drop=True)
display(results)
print('selected model:',results.loc[0,"model"])

,model,validation_results
0,random_forest,0.958730
1,hist_gradient_boosting,0.958730
2,logistic_regression,0.892063


selected model: random_forest


So now we see that the hist gradient boosting and random forest is good for this job ,even that the first dataset was relatively small

# second dataset
the second dataset is big ,and it needs some work because almost all of its data is categorical not like the first one which was all numeric

In [131]:
x3,x4,y3,y4=train_test_split(sec_data.drop(columns=['Churn','customerID']),sec_data["Churn"],test_size=0.2)

In [132]:
y4.value_counts()

Churn
No     1034
Yes     375
Name: count, dtype: int64

Lets play with the data andd see if we can do some feature engineering

In [133]:
x3.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
6682,Male,0,Yes,No,69,Yes,Yes,Fiber optic,No,Yes,No,Yes,Yes,No,Two year,Yes,Mailed check,92.45,6460.55
5964,Female,0,Yes,Yes,49,Yes,No,DSL,Yes,No,Yes,Yes,No,No,Month-to-month,No,Bank transfer (automatic),61.75,3024.15
366,Male,0,Yes,No,23,Yes,No,DSL,Yes,No,No,No,No,Yes,Month-to-month,Yes,Bank transfer (automatic),59.95,1406
3936,Male,0,No,No,9,Yes,No,DSL,Yes,No,Yes,Yes,No,Yes,Month-to-month,Yes,Bank transfer (automatic),68.25,576.95
5618,Male,0,No,No,11,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,20.35,215.25


In [134]:
x3.TotalCharges=pd.to_numeric(x3.TotalCharges,errors='coerce')
median=x3.TotalCharges.median()
x3.TotalCharges.fillna(median,inplace=True)
x4.TotalCharges=pd.to_numeric(x4.TotalCharges,errors='coerce')
x4.TotalCharges.fillna(median,inplace=True)

# x3.TotalCharges=x3.TotalCharges.replace(" ",f"{x3.TotalCharges.median()}").astype(np.float32)

C:\Users\LOQ\AppData\Local\Temp\ipykernel_29264\3570588372.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  x3.TotalCharges.fillna(median,inplace=True)
C:\Users\LOQ\AppData\Local\Temp\ipykernel_29264\3570588372.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, whe

In [135]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_columns = x3.select_dtypes(include=["object", "bool"]).columns.tolist()
numeric_columns = x3.select_dtypes(exclude=["object", "bool"]).columns.tolist()
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("scaler", StandardScaler())
    ]), numeric_columns),
    ("categorical", Pipeline([
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), categorical_columns)
])
def build_pipeline2(model):
    return Pipeline([("preprocessor", preprocessor), ("model", model)])

In [136]:
from sklearn.preprocessing import LabelEncoder

Lencoder=LabelEncoder()
y3=Lencoder.fit_transform(y3)
y4=Lencoder.transform(y4)
y4

array([1, 0, 1, ..., 0, 1, 0], shape=(1409,))

In [137]:

validation_results2=[]
for name, model in models.items():
    candidate=build_pipeline2(model)
    candidate.fit(x3,y3)
    score=accuracy_score(y4,candidate.predict(x4))
    validation_results2.append({"model":name,"validation_results":score})
results2=pd.DataFrame(validation_results2).sort_values("validation_results",ascending=False).reset_index(drop=True)
display(results2)
print('selected model:',results2.loc[0,"model"])

,model,validation_results
0,logistic_regression,0.815472
1,hist_gradient_boosting,0.806955
2,random_forest,0.794180


selected model: logistic_regression
